#  Adapter l'environement

In [1]:
!pip install albumentations -q
import albumentations as A
import cv2
import os
import shutil
import numpy as np
from pathlib import Path

print("✅ Albumentations installé")
print(f"Version : {A.__version__}")

✅ Albumentations installé
Version : 2.0.8


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.listdir('/content/drive/MyDrive'))

Mounted at /content/drive
['Colab Notebooks']


In [ ]:
import os

# Recréer les dossiers
os.makedirs('/content/drive/MyDrive/geoscanner/dataset', exist_ok=True)
os.makedirs('/content/drive/MyDrive/geoscanner/models',  exist_ok=True)

# Uploader kaggle.json
from google.colab import files
files.upload()

# Configurer Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Télécharger le dataset
!kaggle datasets download -d stealthtechnologies/rock-classification \
    -p /content/drive/MyDrive/geoscanner/dataset/

# Décompresser
!unzip /content/drive/MyDrive/geoscanner/dataset/rock-classification.zip \
    -d /content/drive/MyDrive/geoscanner/dataset/

# Vérification
BASE = '/content/drive/MyDrive/geoscanner/dataset/Rock Data'
for split in ['train', 'valid', 'test']:
    path = os.path.join(BASE, split)
    total = sum(
        len(os.listdir(os.path.join(path, c)))
        for c in os.listdir(path)
    )
    print(f"{split:6} → {total} images")

######  Installation Albumentations

In [7]:
!pip install albumentations -q
import albumentations as A
import cv2
import os
import shutil
import numpy as np

print("✅ Albumentations installé")

✅ Albumentations installé


###### Créer la structure

In [8]:
SRC  = '/content/drive/MyDrive/geoscanner/dataset/Rock Data'
DEST = '/content/drive/MyDrive/geoscanner/dataset/Rock Data Augmented'

for split in ['valid', 'test']:
    src_path  = os.path.join(SRC, split)
    dest_path = os.path.join(DEST, split)
    if not os.path.exists(dest_path):
        shutil.copytree(src_path, dest_path)
        print(f"✅ {split}/ copié")

for cls in os.listdir(os.path.join(SRC, 'train')):
    os.makedirs(os.path.join(DEST, 'train', cls), exist_ok=True)

print("✅ Structure créée")

✅ valid/ copié
✅ test/ copié
✅ Structure créée


###### Pipeline d'augmentation

In [9]:
augment = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.5
    ),
    A.HueSaturationValue(
        hue_shift_limit=10,
        sat_shift_limit=20,
        val_shift_limit=10,
        p=0.4
    ),
    A.GaussNoise(var_limit=(10, 50), p=0.3),
    A.Sharpen(alpha=(0.1, 0.3), p=0.3),
    A.Perspective(scale=(0.02, 0.05), p=0.3),
])

print("✅ Pipeline d'augmentation défini")

✅ Pipeline d'augmentation défini


/tmp/ipykernel_1932/582630117.py:16: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 50), p=0.3),


###### Generer les images

In [10]:
TARGET_PER_CLASS = 800

TRAIN_SRC  = os.path.join(SRC,  'train')
TRAIN_DEST = os.path.join(DEST, 'train')

for cls in sorted(os.listdir(TRAIN_SRC)):
    src_cls  = os.path.join(TRAIN_SRC, cls)
    dest_cls = os.path.join(TRAIN_DEST, cls)

    original_images = [
        f for f in os.listdir(src_cls)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    for img_name in original_images:
        shutil.copy2(
            os.path.join(src_cls, img_name),
            os.path.join(dest_cls, img_name)
        )

    current = len(original_images)
    needed  = TARGET_PER_CLASS - current
    generated = 0

    while generated < needed:
        img_name = original_images[generated % current]
        img      = cv2.imread(os.path.join(src_cls, img_name))

        if img is None:
            generated += 1
            continue

        img       = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        augmented = augment(image=img)['image']

        save_path = os.path.join(dest_cls, f"aug_{generated:04d}.jpg")
        cv2.imwrite(save_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))
        generated += 1

    total = len(os.listdir(dest_cls))
    print(f"{cls:25} {current:4d} + {generated:4d} aug = {total:4d}")

print(f"\n✅ Dataset augmenté prêt !")

Basalt                     432 +  368 aug =  800
Clay                       453 +  347 aug =  800
Conglomerate               447 +  353 aug =  800
Diatomite                  405 +  395 aug =  800
Shale-(Mudstone)           396 +  404 aug =  800
Siliceous-sinter           381 +  419 aug =  800
chert                      405 +  395 aug =  800
gypsum                     393 +  407 aug =  800
olivine-basalt             375 +  425 aug =  800

✅ Dataset augmenté prêt !


###### Verification

In [11]:
print("Vérification finale :\n")
for split in ['train', 'valid', 'test']:
    path    = os.path.join(DEST, split)
    classes = sorted(os.listdir(path))
    total   = sum(
        len(os.listdir(os.path.join(path, c)))
        for c in classes
    )
    print(f"{split:6} → {len(classes)} classes — {total} images")

Vérification finale :

train  → 9 classes — 7200 images
valid  → 9 classes — 351 images
test   → 9 classes — 174 images


# Le réentraînement

In [12]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

DEST       = '/content/drive/MyDrive/geoscanner/dataset/Rock Data Augmented'
MODEL_DIR  = '/content/drive/MyDrive/geoscanner/models'
MODEL_PATH = os.path.join(MODEL_DIR, 'resnet50_best_augmented.pth')
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device : {device}")

✅ Device : cuda


In [13]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(os.path.join(DEST, 'train'), train_transform)
valid_ds = datasets.ImageFolder(os.path.join(DEST, 'valid'), val_transform)
test_ds  = datasets.ImageFolder(os.path.join(DEST, 'test'),  val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

print(f"✅ Train  : {len(train_ds)} images")
print(f"✅ Valid  : {len(valid_ds)} images")
print(f"✅ Test   : {len(test_ds)}  images")
print(f"✅ Classes: {train_ds.classes}")

✅ Train  : 7200 images
✅ Valid  : 351 images
✅ Test   : 174  images
✅ Classes: ['Basalt', 'Clay', 'Conglomerate', 'Diatomite', 'Shale-(Mudstone)', 'Siliceous-sinter', 'chert', 'gypsum', 'olivine-basalt']


### Model

In [14]:
model = models.resnet50(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 9)
)

model     = model.to(device)
criterion = nn.CrossEntropyLoss()
print("✅ Modèle prêt")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 114MB/s]


✅ Modèle prêt


### Phase A

In [15]:
model.fc.requires_grad_(True)

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

EPOCHS_A     = 5
best_acc     = 0.0
patience_ctr = 0
PATIENCE     = 4
history      = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print("=" * 55)
print("PHASE A — FC uniquement  lr=0.001")
print("=" * 55)

for epoch in range(EPOCHS_A):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    tl = train_loss / len(train_loader)
    vl = val_loss   / len(valid_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_acc'].append(val_acc)
    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc     = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Epoch {epoch+1:02d}/{EPOCHS_A} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}% ⭐")
    else:
        patience_ctr += 1
        print(f"Epoch {epoch+1:02d}/{EPOCHS_A} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}%")

    if patience_ctr >= PATIENCE:
        print(f"⏹️  Early stopping epoch {epoch+1}")
        break

print(f"\n✅ Phase A terminée — Meilleure acc : {best_acc:.2f}%")

PHASE A — FC uniquement  lr=0.001
Epoch 01/5 — Train: 1.5866 — Val: 1.1990 — Acc: 57.55% ⭐
Epoch 02/5 — Train: 1.2899 — Val: 1.1110 — Acc: 58.69% ⭐
Epoch 03/5 — Train: 1.2324 — Val: 1.0568 — Acc: 63.82% ⭐
Epoch 04/5 — Train: 1.1929 — Val: 1.1018 — Acc: 60.97%
Epoch 05/5 — Train: 1.2030 — Val: 1.1338 — Acc: 60.68%

✅ Phase A terminée — Meilleure acc : 63.82%


###### Phase B

In [16]:
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
model.fc.requires_grad_(True)

optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 0.00001},
    {'params': model.layer4.parameters(), 'lr': 0.00005},
    {'params': model.fc.parameters(),     'lr': 0.0001},
])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=4, factor=0.3
)

EPOCHS_B     = 20
patience_ctr = 0
PATIENCE     = 5

print("=" * 55)
print("PHASE B — layer3+layer4+FC  lr conservateur")
print("=" * 55)

for epoch in range(EPOCHS_B):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            correct  += outputs.max(1)[1].eq(labels).sum().item()
            total    += labels.size(0)

    val_acc = 100. * correct / total
    tl = train_loss / len(train_loader)
    vl = val_loss   / len(valid_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['val_acc'].append(val_acc)
    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc     = val_acc
        patience_ctr = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Epoch {epoch+1:02d}/{EPOCHS_B} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}% ⭐")
    else:
        patience_ctr += 1
        print(f"Epoch {epoch+1:02d}/{EPOCHS_B} — "
              f"Train: {tl:.4f} — Val: {vl:.4f} — "
              f"Acc: {val_acc:.2f}%")

    if patience_ctr >= PATIENCE:
        print(f"⏹️  Early stopping epoch {epoch+1}")
        break

print(f"\n✅ Meilleure accuracy finale : {best_acc:.2f}%")
print(f"✅ Modèle sauvegardé : {MODEL_PATH}")

PHASE B — layer3+layer4+FC  lr conservateur
Epoch 01/20 — Train: 0.8980 — Val: 1.0686 — Acc: 69.52% ⭐
Epoch 02/20 — Train: 0.4981 — Val: 1.0821 — Acc: 72.93% ⭐
Epoch 03/20 — Train: 0.3080 — Val: 1.1681 — Acc: 72.93%
Epoch 04/20 — Train: 0.2097 — Val: 1.3036 — Acc: 71.79%
Epoch 05/20 — Train: 0.1726 — Val: 1.3383 — Acc: 72.08%
Epoch 06/20 — Train: 0.1422 — Val: 1.4473 — Acc: 69.23%
Epoch 07/20 — Train: 0.1171 — Val: 1.6119 — Acc: 69.80%
⏹️  Early stopping epoch 7

✅ Meilleure accuracy finale : 72.93%
✅ Modèle sauvegardé : /content/drive/MyDrive/geoscanner/models/resnet50_best_augmented.pth


###### Test Accuracy

In [17]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Charger le modèle augmenté
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(
    all_labels, all_preds,
    target_names=test_ds.classes
))

# Accuracy manuelle
correct = sum(p == l for p, l in zip(all_preds, all_labels))
print(f"Accuracy test set : {100.*correct/len(all_labels):.2f}%")

                  precision    recall  f1-score   support

          Basalt       0.76      0.87      0.81        15
            Clay       0.57      0.53      0.55        15
    Conglomerate       0.67      0.82      0.74        17
       Diatomite       0.62      0.65      0.64        23
Shale-(Mudstone)       0.58      0.32      0.41        22
Siliceous-sinter       0.72      0.81      0.76        16
           chert       0.50      0.60      0.55        15
          gypsum       0.76      0.81      0.79        27
  olivine-basalt       1.00      0.88      0.93        24

        accuracy                           0.70       174
       macro avg       0.69      0.70      0.69       174
    weighted avg       0.70      0.70      0.69       174

Accuracy test set : 70.11%


In [18]:
from google.colab import files
files.download(MODEL_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>